# Compare baseline and island-inpainted forcing

This notebook compares the canonical ERA5–SCOTIA forcing with the GEBCO-assisted island-inpainted variant. It first diagnoses the Ekman-pumping sources directly, then runs the complete temporal `GlobalRossbyModel.solve()` path for both inputs and compares transport and height output.

The comparison is deliberately read-only: it does not overwrite either forcing dataset or save model output.

In [ ]:
from pathlib import Path
import sys

import dask
import dask.array as da
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
import xarray as xr

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from moc_adjustment_theory import GlobalRossbyModel

ISOBATH_PATH = ROOT / "data/tracked/isobath/global_isobath_GEBCO_1000m.nc"
BASELINE_PATH = ROOT / "data/untracked/forcing/global_ERA5_SCOTIA_forcing.nc"
INPAINTED_PATH = ROOT / "data/untracked/forcing/global_ERA5_SCOTIA_forcing_island_inpainted.nc"
MONTH_SECONDS = 365.25 / 12 * 24 * 60 * 60
G_PRIME = 0.02
EARTH_RADIUS_M = 6_371_000.0

for path in (ISOBATH_PATH, BASELINE_PATH, INPAINTED_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Required input does not exist: {path}")

## Open and verify the forcing pair

Both files must have exactly the same public schema and coordinates. Their only intended difference is the internal-island stress preprocessing recorded in the corrected file's global attributes.

In [ ]:
baseline = xr.open_dataset(BASELINE_PATH, chunks={})
inpainted = xr.open_dataset(INPAINTED_PATH, chunks={})

if set(baseline.data_vars) != set(inpainted.data_vars):
    raise AssertionError("Forcing datasets have different variables")
if baseline.sizes != inpainted.sizes:
    raise AssertionError("Forcing datasets have different dimensions")
for coordinate in ("time", "latitude", "longitude"):
    if not baseline[coordinate].equals(inpainted[coordinate]):
        raise AssertionError(f"Forcing datasets have different {coordinate} coordinates")
xr.testing.assert_identical(baseline.T_N, inpainted.T_N)
print(inpainted.attrs["island_preprocessing"])
print(f"inpainted grid cells: {inpainted.attrs['island_mask_cell_count']:,}")

## Ekman-pumping comparison

The model forcing is $w_{\mathrm{Ek}}=\nabla\cdot\mathbf M_{\mathrm{Ek}}$. The same spherical-grid divergence is evaluated for both files. Temporal maxima expose intermittent land-stress dipoles even when they are absent from a particular monthly snapshot.

In [ ]:
def ekman_divergence(forcing):
    """Return spherical divergence of vector Ekman transport as a DataArray."""
    forcing = forcing.chunk(
        {"time": -1, "latitude": 64, "longitude": 128}
    )
    latitude_radians = np.deg2rad(forcing.latitude.values)
    longitude_radians = np.deg2rad(forcing.longitude.values)
    cosine = np.cos(latitude_radians).astype(np.float32)
    mx = forcing.M_Ek_x.transpose("time", "latitude", "longitude").data
    my = forcing.M_Ek_y.transpose("time", "latitude", "longitude").data
    zonal = da.gradient(mx, longitude_radians, axis=2, edge_order=2)
    meridional = da.gradient(
        my * cosine[None, :, None], latitude_radians, axis=1, edge_order=2
    )
    values = (zonal + meridional) / (
        EARTH_RADIUS_M * cosine[None, :, None]
    )
    return xr.DataArray(
        values,
        dims=("time", "latitude", "longitude"),
        coords={
            "time": forcing.time,
            "latitude": forcing.latitude,
            "longitude": forcing.longitude,
        },
        name="w_Ek",
        attrs={"units": "m s-1"},
    )


w_baseline = ekman_divergence(baseline)
w_inpainted = ekman_divergence(inpainted)
north_atlantic = {
    "latitude": slice(10.0, 45.0),
    "longitude": slice(-40.0, -5.0),
}
with ProgressBar():
    pumping_comparison = xr.Dataset(
        {
            "baseline": abs(w_baseline).max("time"),
            "inpainted": abs(w_inpainted).max("time"),
            "removed": abs(w_baseline - w_inpainted).max("time"),
        }
    ).sel(**north_atlantic).compute()

In [ ]:
fig, axes = plt.subplots(
    1, 3, figsize=(14, 4), constrained_layout=True, sharex=True, sharey=True
)
for axis, name, title in zip(
    axes,
    ("baseline", "inpainted", "removed"),
    ("Baseline peak |w_Ek|", "Inpainted peak |w_Ek|", "Peak removed signal"),
):
    pumping_comparison[name].plot.pcolormesh(
        ax=axis,
        x="longitude",
        y="latitude",
        norm=LogNorm(vmin=1e-7, vmax=5e-4),
        cmap="magma",
        add_colorbar=True,
    )
    axis.set_title(title)
plt.show()

for name in ("baseline", "inpainted", "removed"):
    print(name, float(pumping_comparison[name].max()), "m s-1")

## Run the complete model for both forcings

Both model calls use the canonical reduced gravity and monthly sample interval. Construction remains lazy; one combined Dask computation below evaluates the low-dimensional transport diagnostics and the North Atlantic height snapshot while reusing each solution's internal graph.

In [ ]:
isobath = xr.open_dataset(ISOBATH_PATH)
model = GlobalRossbyModel(isobath, g_prime=G_PRIME)
baseline_solution = model.solve(
    baseline, sample_spacing_seconds=MONTH_SECONDS
)
inpainted_solution = model.solve(
    inpainted, sample_spacing_seconds=MONTH_SECONDS
)

snapshot_selection = {
    "region": "north_atlantic",
    "time": "2012-05-01",
}
with dask.config.set(scheduler="threads", num_workers=4), ProgressBar():
    (
        baseline_transport,
        inpainted_transport,
        baseline_height,
        inpainted_height,
    ) = dask.compute(
        baseline_solution[["T", "T_Ek", "T_g", "h_e"]],
        inpainted_solution[["T", "T_Ek", "T_g", "h_e"]],
        baseline_solution.h.sel(**snapshot_selection),
        inpainted_solution.h.sel(**snapshot_selection),
    )

## Transport impact

RMS and peak changes are reported relative to the baseline anomaly signal. The latitude profiles show whether island corrections remain confined to narrow rows or alter the basin-scale transport solution.

In [ ]:
def finite_rms(values):
    """Return the root-mean-square of all finite values in an array."""
    values = np.asarray(values)
    finite = np.isfinite(values)
    return float(np.sqrt(np.mean(values[finite] ** 2)))


records = []
for diagnostic in ("T", "T_Ek", "T_g"):
    for region in baseline_transport.region.values:
        original = baseline_transport[diagnostic].sel(region=region)
        corrected = inpainted_transport[diagnostic].sel(region=region)
        difference = corrected - original
        original_rms = finite_rms(original)
        difference_rms = finite_rms(difference)
        records.append(
            {
                "diagnostic": diagnostic,
                "region": str(region),
                "baseline_rms_Sv": original_rms / 1e6,
                "change_rms_Sv": difference_rms / 1e6,
                "change_percent": 100.0 * difference_rms / original_rms,
                "peak_change_Sv": float(abs(difference).max(skipna=True)) / 1e6,
            }
        )
transport_summary = pd.DataFrame.from_records(records)
transport_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for region in baseline_transport.region.values:
    difference = (
        inpainted_transport.T.sel(region=region)
        - baseline_transport.T.sel(region=region)
    ) / 1e6
    rms_profile = np.sqrt((difference**2).mean("time"))
    rms_profile.plot(ax=axes[0], label=str(region))
axes[0].set_title("RMS total-transport change")
axes[0].set_ylabel("Sv")
axes[0].legend(fontsize=8)

latitude = 26.5
baseline_series = baseline_transport.T.sel(
    region="north_atlantic", latitude=latitude
) / 1e6
inpainted_series = inpainted_transport.T.sel(
    region="north_atlantic", latitude=latitude
) / 1e6
baseline_series.plot(ax=axes[1], label="baseline")
inpainted_series.plot(ax=axes[1], label="island-inpainted")
axes[1].set_title(f"North Atlantic total transport at {latitude}°N")
axes[1].set_ylabel("Sv")
axes[1].legend()
plt.show()

## Height-field comparison

The May 2012 North Atlantic snapshot is the case in which island-generated latitude bands were first identified. Identical colour limits make amplitude changes visible; the third panel isolates the correction.

In [ ]:
height_difference = inpainted_height - baseline_height
combined = np.concatenate(
    (
        baseline_height.values[np.isfinite(baseline_height.values)],
        inpainted_height.values[np.isfinite(inpainted_height.values)],
    )
)
limit = float(np.nanpercentile(np.abs(combined), 99.5))
difference_limit = float(np.nanpercentile(np.abs(height_difference), 99.5))

fig, axes = plt.subplots(
    1, 3, figsize=(14, 4), constrained_layout=True, sharex=True, sharey=True
)
for axis, field, title, bound in (
    (axes[0], baseline_height, "Baseline h", limit),
    (axes[1], inpainted_height, "Island-inpainted h", limit),
    (axes[2], height_difference, "Inpainted minus baseline", difference_limit),
):
    field.plot.pcolormesh(
        ax=axis,
        x="longitude",
        y="latitude",
        cmap="RdBu_r",
        vmin=-bound,
        vmax=bound,
    )
    axis.set_title(title)
plt.show()

print(
    {
        "baseline_height_rms_m": finite_rms(baseline_height),
        "inpainted_height_rms_m": finite_rms(inpainted_height),
        "height_change_rms_m": finite_rms(height_difference),
        "baseline_height_peak_m": float(abs(baseline_height).max(skipna=True)),
        "inpainted_height_peak_m": float(abs(inpainted_height).max(skipna=True)),
    }
)

## Interpretation

The forcing and model comparisons above distinguish three effects:

1. removal of localized island-generated Ekman-pumping dipoles;
2. changes to section-integrated Ekman transport where ERA5 land stress previously entered the zonal integral; and
3. the dynamically propagated response in total transport, boundary height, and the spatial height field.

A production preprocessing change should preserve the same checks and add a sensitivity comparison for the island-mask definition and interpolation method.